# Kaggle GPU embed keyframes — PE-Core-bigG-14-448 (ứng viên A2)

Encode toàn bộ keyframe của **873 video** (CÙNG bộ keyframe dataset đã dùng cho BEiT-3 và
SigLIP 2, xem `notebooks/kaggle_embed_keyframes.ipynb` / `notebooks/kaggle_embed_siglip2.ipynb`)
bằng PE-Core-bigG-14-448 (`timm/PE-Core-bigG-14-448`, `PeCoreEncoder` ở
`app/encoders/pe_core.py`), xuất **artifact vector** ra `/kaggle/working/embed_out_pecore/`.
Đây là candidate A2 của `.claude/plans/encoder-swap-siglip2-pecore.plan.md` §T7 -- chính
encoder MERVIN (arXiv 2605.16120) dùng để đạt 79/88 điểm vòng sơ tuyển.

**Notebook này KHÔNG ingest vào Milvus/MongoDB nào cả** — cùng nguyên tắc phân vai với hai
notebook trước: *Kaggle chỉ làm việc cần GPU hoặc cần ảnh, local giữ toàn bộ sự thật về
metadata*. Import artifact này vào DB local (qua cờ `--encoder pe_core` của
`scripts/import_embed_out.py`) là bước riêng, **CHƯA implement** tại thời điểm viết notebook
này -- `app/encoders/factory.py` hiện chỉ dispatch `("beit3", "siglip2")`.

## License — đã xác nhận lại 2026-08-14 (không chỉ tin lời đã nghe)

Fetch trực tiếp `https://huggingface.co/api/models/timm/PE-Core-bigG-14-448` và
`.../raw/main/open_clip_config.json`: **`apache-2.0`**, không có ràng buộc FAIR-NC/
non-commercial nào. Đóng hẳn nghi ngờ ở `docs/research-encoder-vietnamese.md` §9 Q1 ("Meta có
thể là FAIR-NC"). `embedding_dim` xác nhận là **1280** (không phải suy đoán ~1.8B tham số
trong research doc cũ), vision tower là `vit_pe_core_gigantic_patch14_448` -- **timm-backed
thật**, xác nhận đúng lý do `pyproject.toml` cần pin `timm>=1.0.19`.

## ⚠️ Khác biệt quan trọng so với notebook SigLIP 2 (A1) — đọc trước khi chạy

`open_clip==3.3.0`'s `create_model_and_transforms("hf-hub:<repo>")` **KHÔNG cho pin revision**
— đã verify trực tiếp trong source (`src/open_clip/factory.py`, `src/open_clip/pretrained.py`):
hàm nội bộ `download_pretrained_from_hf()` CÓ tham số `revision`, nhưng nhánh `hf-hub:` của
factory không bao giờ truyền nó vào. Khác hẳn `SigLip2Encoder` (`AutoModel.from_pretrained(...,
revision=...)` ghim cứng được). Hệ quả:

- `PeCoreEncoder` (và `configs/model/pe_core.yaml`) **không có tham số `hf_revision`** — thêm
  vào mà không dùng được sẽ là kiểu "hỏng âm thầm" CLAUDE.md §3 cảnh báo.
- Notebook này **resolve commit SHA hiện tại của "main" qua HF API ngay lúc chạy** rồi ghi vào
  `manifest.json` như thông tin xuất xứ (`hf_revision`, `hf_revision_pinned: false`) -- không
  phải một pin có hiệu lực chặn.
- Cổng an toàn thật sự vẫn là **image + text probe** so khớp cosine (mục 11) -- nếu weight
  trên Hub đổi giữa lúc Kaggle chạy và lúc local re-encode probe, probe sẽ tự phát hiện ra
  bằng cosine thấp, dù không có gì "pin" được từ đầu.

## Dùng cả 2 GPU (Kaggle "T4 x2")

`NUM_SHARDS = 2` trong cell cấu hình (mục 4) — smoke test (mục 8) và full run (mục 9) đều
launch **`NUM_SHARDS` process song song bằng `subprocess.Popen`**, mỗi process ghim
`--device cuda:{shard_id}` riêng, chia danh sách video **xen kẽ** (`[shard_id::num_shards]` —
không chia khối liên tiếp, vì mật độ keyframe/video lệch nhiều giữa các batch, chia khối sẽ
dồn hết video nặng vào 1 GPU). Cả hai process ghi vào cùng `vectors/` an toàn vì tập
`video_id` giữa các shard **không giao nhau**. Nếu Notebook Settings không có "T4 x2" (chỉ
"T4 x1" hoặc "P100"), đổi `NUM_SHARDS = 1` — cell cấu hình sẽ tự raise nếu
`NUM_SHARDS` > số GPU thật `torch.cuda.device_count()` phát hiện được.

## Đầu ra

```
embed_out_pecore/
├── manifest.json                  # manifest v2: model_id, hf_revision (không pin được, xem trên),
│                                    # weight_files_sha256, hf_revision_pinned: false, ...
├── probe/
│   ├── probe_texts.json            # 20 câu cố định (giống hệt bộ đã dùng cho BEiT-3/SigLIP 2)
│   ├── probe_text_vectors.npy      # (20, 1280) float32
│   ├── image_probe.json            # 16 keyframe_id cố định, chọn trải đều qua các batch
│   └── image_probe_vectors.npy     # (16, 1280) float32
└── vectors/
    ├── L29_V012.npy                 # (n_kf, 1280) float32, ĐÃ L2-normalize
    ├── L29_V012.jsonl               # n_kf dòng, THỨ TỰ KHỚP HÀNG của .npy
    └── ...
```

`keyframe_id = "{video_id}_f{frame_idx}"` — CÙNG hợp đồng với artifact BEiT-3/SigLIP 2
(`.claude/plans/kaggle-offline-embed.plan.md` §3).

## Thứ tự chạy

1. Kiểm tra GPU (T4 hay P100 — ảnh hưởng throughput và độ trôi số fp16)
2. Clone repo qua Kaggle Secret `GITHUB_TOKEN`
3. Cài `open-clip-torch==3.3.0` + `timm>=1.0.19` (kernel mặc định, không env riêng)
4. Cấu hình — mọi hằng số dồn về đúng một cell
5. Kiểm tra vision tower thật sự timm-backed (KHÁC A1: bẫy riêng của open_clip)
6. Kiểm kê bộ keyframe — fail loud nếu lệch
7. Sinh script embed ra `/kaggle/working/embed_pecore.py`
8. **Smoke test ~1.000 keyframe** — mô hình nặng nhất trong plan, đo throughput thật trước khi treo job dài
9. **Chạy full 873 video** — chỉ bấm sau khi mục 8 xanh VÀ đã kiểm ngân sách 30h/tuần còn lại (trừ đi giờ A1 đã dùng)
10. Kiểm tra output
11. Image probe + text probe
12. Sinh `manifest.json` (v2, có `hf_revision_pinned: false`)
13. Hướng dẫn mang về local

## Cảnh báo tốn tài nguyên & ngân sách GPU

Mục 9 là cell nặng nhất, và đây là **candidate nặng nhất trong plan** (~1.8B tham số vision
tower, ảnh 448px, so với so400m's ~400M @ 384px). Kaggle giới hạn **30 giờ GPU/tuần CHUNG cho
cả A1 và A2** -- kiểm tra ngân sách còn lại sau khi A1 đã chạy trước khi bấm mục 9. Mục 8
(smoke test) cho số đo thật đầu tiên; KHÔNG suy đoán từ số của A1 (kiến trúc khác hẳn).

**Chạy `NUM_SHARDS=2` (T4 x2) tốn quota GPU theo GIỜ-GPU, không phải giờ ĐỒNG HỒ**: 1 giờ
đồng hồ chạy song song 2 GPU trừ **2 giờ** vào ngân sách 30h/tuần, không phải 1 giờ. Rút ngắn
thời gian chờ ~2x, nhưng KHÔNG tiết kiệm quota so với chạy tuần tự trên 1 GPU.

## Việc cần làm TRƯỚC khi mở notebook này trên Kaggle

1. **Commit + push nhánh chứa code này lên GitHub** — `app/encoders/pe_core.py`,
   `configs/model/pe_core.yaml` phải có mặt trên nhánh mà biến `REPO_BRANCH` (mục 2) trỏ tới.
2. **Kaggle Secret `GITHUB_TOKEN`** (repo `tiendze/AIC-Video-Retrieval` là private).
3. **Add Kaggle Dataset keyframe** — CÙNG dataset đã dùng cho hai notebook trước, cấu trúc
   `<batch>/<video_id>/<video_id>_<frame_idx>.jpg`.
4. **Bật Internet + GPU accelerator** trong Notebook Settings.

## Ngoài phạm vi notebook này

- **Import vào Milvus/MongoDB local** — cần `--encoder pe_core` trong
  `scripts/import_embed_out.py` + dispatch `"pe_core"` trong `app/encoders/factory.py`
  (plan §T1/§T5), **chưa implement** tại thời điểm viết notebook này.
- **Kênh eval `pecore_en` trong `eval/run.py`** (plan §T9) — chưa implement.


## 1. Kiểm tra GPU

Ghi lại tên GPU hiện ra ở dòng `nvidia-smi` (T4 hay P100) — ảnh hưởng cả throughput lẫn độ trôi số fp16 giữa Kaggle và RTX 3070 local.

In [ ]:
!nvidia-smi


## 2. Clone repo qua Kaggle Secret `GITHUB_TOKEN`

Yêu cầu: đã thêm Kaggle Secret tên `GITHUB_TOKEN` (Add-ons → Secrets) chứa Personal Access
Token có quyền đọc repo `tiendze/AIC-Video-Retrieval` (private). Clone không phải để lấy code
mới — mà vì cần `app/encoders/pe_core.py` đã có sẵn, chép lại vào cell là vô nghĩa.

In [ ]:
from pathlib import Path

from kaggle_secrets import UserSecretsClient

github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")

REPO_BRANCH = "main"  # DOI neu ban commit code nay len nhanh khac -- phai PUSH len GitHub
                       # truoc khi mo notebook nay (app/encoders/pe_core.py, configs/model/
                       # pe_core.yaml phai co mat tren nhanh nay)
REPO_DIR = Path("/kaggle/working/AIC-Video-Retrieval")
if not REPO_DIR.exists():
    clone_url = f"https://{github_token}@github.com/tiendze/AIC-Video-Retrieval.git"
    !git clone --branch {REPO_BRANCH} {clone_url} {REPO_DIR}
else:
    print(f"{REPO_DIR} da ton tai, bo qua clone (session duoc resume).")

%cd {REPO_DIR}
!git log --oneline -3


## 3. Cài dependency (kernel mặc định — KHÔNG cần env Python 3.10 riêng)

`PeCoreEncoder` cần `torch` (đã có sẵn trên Kaggle, KHÔNG cài đè — bản Kaggle đã khớp CUDA
driver của instance) + `open_clip` + `timm>=1.0.19` (cài mới, Kaggle mặc định thường có timm
cũ hơn) + `Pillow` (thường có sẵn). Không `pip install -e .` — `pyproject.toml` khai
`requires-python = ">=3.10,<3.11"`, kernel mặc định Kaggle là 3.11/3.12 nên pip sẽ từ chối;
chỉ cần thêm repo vào `sys.path`.

⚠️ **Bẫy shadow module** (giống notebook SigLIP 2): repo có package `datasets/` trùng tên thư
viện HF `datasets`. Script embed ở mục 7 dùng `sys.path.append(...)` (thêm vào **cuối**), không
`sys.path.insert(0, ...)`.

In [ ]:
!pip install -q -U "open-clip-torch==3.3.0" "timm>=1.0.19"

import open_clip
import timm
import torch

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("open_clip:", open_clip.__version__)
print("timm:", timm.__version__)

_timm_major_minor = tuple(int(p) for p in timm.__version__.split(".")[:2])
if _timm_major_minor < (1, 0):
    raise RuntimeError(
        f"timm {timm.__version__} qua cu -- can >=1.0.19 de load vision tower "
        "vit_pe_core_gigantic_patch14_448 (xem app/encoders/pe_core.py module docstring)."
    )


## 4. Cấu hình

Mọi hằng số dồn về đúng một cell (cùng quyết định gộp đã chốt ở
`.claude/plans/kaggle-offline-embed.plan.md` §4/§5, giữ nguyên cho notebook này). **Không có
`HF_REVISION`** ở đây -- xem cảnh báo ở đầu notebook: `open_clip`'s hf-hub loader không cho pin
revision, nên không có gì để cấu hình theo hướng đó.

In [ ]:
import json
from pathlib import Path

# Cau truc bat buoc: <batch>/<video_id>/<video_id>_<frame_idx>.jpg (3 tang) -- CUNG bo
# keyframe dataset da dung cho BEiT-3 va SigLIP 2.
KF_ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/embed_out_pecore")
VECTORS_DIR = OUT_DIR / "vectors"
PROBE_DIR = OUT_DIR / "probe"

CHUNK_SIZE = 128        # so anh mo cung luc moi video truoc khi dua vao encoder (bo nho) --
                        # thap hon SigLIP 2 (256) vi anh 448px nang hon nhieu so 384px
BATCH_SIZE = 8          # batch GPU that su truyen cho PeCoreEncoder -- CO Y bao thu hon ca
                        # SigLIP 2 (BATCH_SIZE=16): bigG la candidate nang nhat plan (~1.8B
                        # tham so vision tower vs so400m's ~400M, anh 448px vs 384px). Don
                        # bay dau tien khi can tang throughput la PREPROCESS_WORKERS, khong
                        # phai BATCH_SIZE (bai hoc do duoc tu ClipEncoder 2026-08-12 va
                        # SigLip2Encoder 2026-08-14) -- chi tang BATCH_SIZE sau khi da thu
                        # PREPROCESS_WORKERS va thay GPU van ranh.
PREPROCESS_WORKERS = 8

NUM_SHARDS = 2  # Kaggle T4 x2 -- 1 process per GPU, video_id list interleaved (shard_id::
                # num_shards) khong chia khoi lien tiep: video_ids dang sort theo batch prefix
                # (L22_..., L23_...), va mat do keyframe/video lech rat nhieu giua cac batch
                # (kaggle-offline-embed.plan.md Section 2) -- chia khoi se don het video nang
                # vao 1 shard. PREPROCESS_WORKERS o tren la SO LUONG MOI PROCESS (khong chia
                # cho NUM_SHARDS) -- 2 process x 8 worker = 16 thread CPU tranh nhau; neu thay
                # CPU nghen (Task Manager/nvidia-smi cho thay GPU cho), giam PREPROCESS_WORKERS
                # truoc khi giam BATCH_SIZE, dung nguyen tac do that cua config nay.

HF_REPO_ID = "timm/PE-Core-bigG-14-448"
MODEL_ID = "pe_core_bigg14_448"  # phai khop configs/model/pe_core.yaml (local)
EMBEDDING_DIM = 1280  # verified 2026-08-14 tu open_clip_config.json model_cfg.embed_dim --
                      # chi dung de canh bao lech, khong ep buoc

EXPECTED_NUM_VIDEOS = 873
EXPECTED_NUM_KEYFRAMES = 221_903  # cung bo dataset da do cho BEiT-3/SigLIP 2

IMAGE_PROBE_COUNT = 16

# Giu NGUYEN VAN cung 20 cau probe text da dung cho BEiT-3/SigLIP 2 -- de so sanh cong bang
# "cau nao cac encoder xu ly khac nhau nhieu nhat" duoc, khong phai vi noi dung cau quan trong.
PROBE_TEXTS = [
    "một người đàn ông đang phát biểu trước ống kính",
    "cảnh quay từ trên cao xuống một con đường đông xe cộ",
    "phóng viên đứng trước tòa nhà chính phủ đưa tin",
    "hai người đang bắt tay nhau trong một sự kiện",
    "một chiếc xe cứu hỏa đang di chuyển trên đường phố",
    "đám đông tụ tập tại một quảng trường lớn",
    "một đầu bếp đang chế biến món ăn trong nhà bếp",
    "trận bóng đá đang diễn ra trên sân vận động",
    "một chiếc máy bay cất cánh từ đường băng",
    "người dẫn chương trình đang giới thiệu khách mời",
    "a man is speaking in front of a camera",
    "aerial view of a crowded street with many cars",
    "a reporter standing in front of a government building",
    "two people shaking hands at an event",
    "a fire truck moving through the street",
    "a crowd gathered in a large square",
    "a chef preparing food in a kitchen",
    "a football match taking place in a stadium",
    "an airplane taking off from a runway",
    "a host introducing a guest on a talk show",
]
assert len(PROBE_TEXTS) == 20

# PE-Core la English-only (custom_text CLIP BPE tokenizer, context_length=72) -- khac SigLIP 2
# (Gemma tokenizer, 109 ngon ngu). 10 cau tieng Viet o tren VAN duoc encode (khong loi), nhung
# vector cua chung khong dang tin cay ve mat ngu nghia -- giu lai chi de dong bo bo cau hoi
# giua 3 encoder, KHONG dung ket qua probe tieng Viet cua PE-Core de danh gia chat luong.

print("KF_ROOT:", KF_ROOT)
print("OUT_DIR:", OUT_DIR)
print("CHUNK_SIZE:", CHUNK_SIZE, "| BATCH_SIZE:", BATCH_SIZE, "| PREPROCESS_WORKERS:", PREPROCESS_WORKERS)
print("NUM_SHARDS:", NUM_SHARDS)
print("HF_REPO_ID:", HF_REPO_ID)
print("MODEL_ID:", MODEL_ID)

_available_gpus = torch.cuda.device_count()
print(f"torch.cuda.device_count(): {_available_gpus}")
if _available_gpus < NUM_SHARDS:
    raise RuntimeError(
        f"NUM_SHARDS={NUM_SHARDS} nhung chi thay {_available_gpus} GPU -- kiem tra lai "
        "Notebook Settings > Accelerator (can chon 'GPU T4 x2', khong phai 'GPU T4 x1' hay P100)."
    )


## 5. Kiểm tra vision tower thật sự timm-backed (KHÁC A1)

Bẫy riêng của candidate này (plan §T7 validate step): nếu `open_clip` load nhầm/fallback sang
một stub không phải timm, `embedding_dim` vẫn có thể khớp 1280 một cách trùng hợp trong khi
vision tower thực tế sai. Build thử `PeCoreEncoder` một lần ở đây (tốn một lượt tải checkpoint
~vài GB, nhưng cache lại cho các bước sau) và in `model.visual.__class__` ra để xác nhận.

In [ ]:
import sys

sys.path.append(str(REPO_DIR))  # append, KHONG insert(0, ...) -- xem canh bao muc 3
from app.encoders.pe_core import PeCoreEncoder  # noqa: E402

_device = "cuda" if torch.cuda.is_available() else "cpu"
_probe_encoder = PeCoreEncoder(hf_repo_id=HF_REPO_ID, device=_device, batch_size=2, model_id=MODEL_ID)

print("model.visual class:", type(_probe_encoder._model.visual))
print("model.visual module:", type(_probe_encoder._model.visual).__module__)
print("embedding_dim:", _probe_encoder.embedding_dim)

_is_timm_backed = "timm" in type(_probe_encoder._model.visual).__module__.lower() or hasattr(
    _probe_encoder._model.visual, "trunk"
)
if not _is_timm_backed:
    raise AssertionError(
        "model.visual khong co dau hieu la timm-backed tower -- dung lai, kiem tra lai "
        "timm/open_clip version truoc khi chay smoke test."
    )
if _probe_encoder.embedding_dim != EMBEDDING_DIM:
    raise AssertionError(
        f"embedding_dim thuc te {_probe_encoder.embedding_dim} != ky vong {EMBEDDING_DIM}"
    )
print("OK: vision tower la timm-backed, embedding_dim khop ky vong.")

del _probe_encoder
torch.cuda.empty_cache() if torch.cuda.is_available() else None


## 6. Kiểm kê bộ keyframe (kernel mặc định, không cần GPU)

Chạy trước khi sinh/chạy script embed — nếu có file lạc thư mục, trùng `(video_id, frame_idx)`,
hay tên file không parse được, **dừng ở đây**. Cùng logic kiểm kê với hai notebook trước (cùng
dataset, cùng regex).

In [ ]:
import re
from collections import Counter

_FILENAME_RE = re.compile(r"^(L\d+_V\d+)_(\d+)\.(jpg|jpeg|png)$", re.IGNORECASE)

image_paths = sorted(
    p for p in KF_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png")
)
print(f"Tim thay {len(image_paths)} file anh duoi {KF_ROOT}")
if not image_paths:
    raise FileNotFoundError(
        f"Khong tim thay anh keyframe nao duoi {KF_ROOT} -- kiem tra da Add dung Kaggle "
        "Dataset keyframe vao notebook chua (CUNG dataset da dung cho hai notebook truoc)."
    )

parse_errors: list[str] = []
seen: dict[tuple[str, int], Path] = {}

for path in image_paths:
    match = _FILENAME_RE.match(path.name)
    if not match:
        parse_errors.append(f"ten file khong khop pattern: {path}")
        continue
    video_id, frame_idx_str, _ext = match.groups()
    if path.parent.name != video_id:
        parse_errors.append(
            f"video_id trong ten file ({video_id}) khong khop thu muc cha ({path.parent.name}): {path}"
        )
        continue
    key = (video_id, int(frame_idx_str))
    if key in seen:
        parse_errors.append(f"trung (video_id, frame_idx)={key}: {path} va {seen[key]}")
        continue
    seen[key] = path

if parse_errors:
    print(f"\n{len(parse_errors)} loi kiem ke -- in toi da 50 dong dau:")
    for line in parse_errors[:50]:
        print(" -", line)
    raise ValueError(
        f"{len(parse_errors)} loi kiem ke bo keyframe -- dung lai, khong chay embed tren du "
        "lieu chua xac thuc."
    )

video_ids = sorted({video_id for video_id, _ in seen})
per_video_counts = Counter(video_id for video_id, _ in seen)

print(f"Kiem ke OK: {len(seen)} keyframe hop le, {len(video_ids)} video duy nhat, 0 loi.")
print(f"Ky vong: {EXPECTED_NUM_KEYFRAMES} keyframe / {EXPECTED_NUM_VIDEOS} video.")
if len(seen) != EXPECTED_NUM_KEYFRAMES or len(video_ids) != EXPECTED_NUM_VIDEOS:
    print(
        "CANH BAO: so do khong khop ky vong -- dataset tren Kaggle co the khac ban da do. "
        "Khong raise (du lieu van hop le voi chinh so lieu cua no), nhung kiem tra lai truoc "
        "khi tin manifest.json o buoc cuoi."
    )

batch_counts = Counter(video_id.split("_V", 1)[0] for video_id in video_ids)
for batch, count in sorted(batch_counts.items()):
    print(f"  {batch}: {count} video")


## 7. Sinh script embed (`%%writefile`)

Cùng lý do như hai notebook trước: code embed cần chạy như script riêng (import
`app.encoders.pe_core` qua `sys.path`, không phải qua magic cell). Script tự đảm bảo:

- **Sort tất định** theo `(video_id, frame_idx)` — giữ nguyên `τ < t` của DANTE.
- **`keyframe_id = "{video_id}_f{frame_idx}"`** — CÙNG hợp đồng với artifact BEiT-3/SigLIP 2.
- **Resume theo video**, ghi file tạm rồi `os.replace`.
- **`--limit-keyframes`** — mật độ keyframe/video lệch nhiều giữa các batch, smoke test theo
  SỐ KEYFRAME cho ước lượng công bằng hơn theo số video.
- **`sys.path.append`** (thêm vào cuối), không `insert(0, ...)`.
- **KHÔNG có `--hf-revision`** — `PeCoreEncoder` không nhận tham số này (xem cảnh báo đầu
  notebook).

In [ ]:
%%writefile /kaggle/working/embed_pecore.py
"""Embed AIC 2026 keyframes with PE-Core-bigG-14-448 on Kaggle GPU -- candidate A2
of the encoder-swap plan (.claude/plans/encoder-swap-siglip2-pecore.plan.md T7).

Reads pre-extracted keyframes from a Kaggle Dataset (<batch>/<video_id>/<video_id>_<frame_idx>.jpg)
-- the SAME dataset already used for the BEiT-3/SigLIP 2 embed runs -- encodes each image with
PeCoreEncoder (app/encoders/pe_core.py, cloned from the repo, imported via sys.path, NOT
installed as a package), and writes one (.npy, .jsonl) pair per video_id under
--out-dir/vectors/. Row i of the .jsonl always corresponds to row i of the .npy, and
keyframe_id is "{video_id}_f{frame_idx}" -- the SAME sort/id contract as the BEiT-3/SigLIP 2
artifacts (.claude/plans/kaggle-offline-embed.plan.md Section 3).

No --hf-revision flag: PeCoreEncoder only takes hf_repo_id -- open_clip's hf-hub loader cannot
pin a revision at load time (see app/encoders/pe_core.py module docstring).

--num-shards/--shard-id: this script is invoked once PER GPU by the notebook (Kaggle T4 x2),
each with a disjoint --device and a disjoint slice of video_ids ([shard_id::num_shards] on the
sorted list -- interleaved, not a contiguous block, since keyframe density varies heavily
across batches). All shards write into the SAME --out-dir/vectors/ concurrently; this is safe
because video_id sets never overlap across shards, so no two processes ever touch the same
.npy/.jsonl pair.

Generated by notebooks/kaggle_embed_pecore.ipynb (%%writefile) -- not tracked in git.
"""
from __future__ import annotations

import argparse
import json
import os
import re
import sys
import time
from pathlib import Path

import numpy as np
from PIL import Image

_FILENAME_RE = re.compile(r"^(L\d+_V\d+)_(\d+)\.(jpg|jpeg|png)$", re.IGNORECASE)
_NORM_TOLERANCE = 1e-3


def _parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--kf-root", type=Path, required=True)
    parser.add_argument("--out-dir", type=Path, required=True)
    parser.add_argument("--repo-dir", type=Path, required=True)
    parser.add_argument("--hf-repo-id", required=True)
    parser.add_argument("--model-id", required=True)
    parser.add_argument("--chunk-size", type=int, default=128)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--preprocess-workers", type=int, default=8)
    parser.add_argument("--device", default="cuda")
    parser.add_argument(
        "--limit-videos", type=int, default=0, help="0 = khong gioi han; >0 chi cho smoke test"
    )
    parser.add_argument(
        "--limit-keyframes",
        type=int,
        default=0,
        help="0 = khong gioi han; >0 dung SAU khi vuot nguong nay (don vi video, khong cat "
        "giua video) -- dung cho smoke test theo so keyframe thay vi so video",
    )
    parser.add_argument(
        "--num-shards", type=int, default=1, help="Tong so process chay song song (vd. 2 cho T4 x2)"
    )
    parser.add_argument(
        "--shard-id", type=int, default=0, help="Chi so shard nay, 0-indexed, < num-shards"
    )
    return parser.parse_args()


def discover_keyframes(kf_root: Path) -> dict[str, list[tuple[int, Path]]]:
    """Parse, validate, and group keyframe files. Raises on any parse/duplicate error."""
    image_paths = sorted(
        p for p in kf_root.rglob("*") if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png")
    )
    if not image_paths:
        raise FileNotFoundError(f"Khong tim thay anh keyframe nao duoi {kf_root}")

    errors: list[str] = []
    seen: dict[tuple[str, int], Path] = {}
    grouped: dict[str, list[tuple[int, Path]]] = {}

    for path in image_paths:
        match = _FILENAME_RE.match(path.name)
        if not match:
            errors.append(f"ten file khong khop pattern: {path}")
            continue
        video_id, frame_idx_str, _ext = match.groups()
        if path.parent.name != video_id:
            errors.append(f"video_id ({video_id}) khong khop thu muc cha ({path.parent.name}): {path}")
            continue
        frame_idx = int(frame_idx_str)
        key = (video_id, frame_idx)
        if key in seen:
            errors.append(f"trung (video_id, frame_idx)={key}: {path} va {seen[key]}")
            continue
        seen[key] = path
        grouped.setdefault(video_id, []).append((frame_idx, path))

    if errors:
        for line in errors[:50]:
            print(" -", line, file=sys.stderr)
        raise ValueError(f"{len(errors)} loi kiem ke bo keyframe -- dung lai truoc khi embed.")

    for video_id in grouped:
        grouped[video_id].sort(key=lambda item: item[0])  # frame_idx tang dan -> giu tau < t
    return grouped


def _select_video_ids(
    video_ids: list[str],
    grouped: dict[str, list[tuple[int, Path]]],
    shard_id: int,
    num_shards: int,
    limit_videos: int,
    limit_keyframes: int,
) -> list[str]:
    # Shard TRUOC khi limit: video_ids da sort theo batch prefix (L22_..., L23_...), va mat do
    # keyframe/video lech rat nhieu giua cac batch (kaggle-offline-embed.plan.md Section 2) --
    # [shard_id::num_shards] xen ke qua danh sach da sort thay vi chia khoi lien tiep, tranh don
    # het video nang vao 1 shard trong khi shard kia video nhe.
    video_ids = video_ids[shard_id::num_shards]
    if limit_videos > 0:
        video_ids = video_ids[:limit_videos]
    if limit_keyframes > 0:
        selected: list[str] = []
        total = 0
        for video_id in video_ids:
            selected.append(video_id)
            total += len(grouped[video_id])
            if total >= limit_keyframes:
                break
        video_ids = selected
    return video_ids


def _is_video_done(out_dir: Path, video_id: str, expected_rows: int) -> bool:
    npy_path = out_dir / "vectors" / f"{video_id}.npy"
    jsonl_path = out_dir / "vectors" / f"{video_id}.jsonl"
    if not (npy_path.exists() and jsonl_path.exists()):
        return False
    with jsonl_path.open(encoding="utf-8") as f:
        num_lines = sum(1 for _ in f)
    if num_lines != expected_rows:
        return False
    vectors = np.load(npy_path)
    return vectors.shape[0] == expected_rows


def _embed_video(encoder, frame_records: list[tuple[int, Path]], chunk_size: int) -> np.ndarray:
    chunks = [frame_records[i : i + chunk_size] for i in range(0, len(frame_records), chunk_size)]
    vector_chunks = []
    for chunk in chunks:
        images = [Image.open(path) for _, path in chunk]
        try:
            vector_chunks.append(encoder.encode_image(images))
        finally:
            for image in images:
                image.close()
    return np.concatenate(vector_chunks, axis=0).astype(np.float32)


def _assert_normalized(vectors: np.ndarray, video_id: str) -> None:
    if vectors.shape[0] == 0:
        return
    sample_size = min(8, vectors.shape[0])
    sample_idx = np.random.default_rng(0).choice(vectors.shape[0], size=sample_size, replace=False)
    norms = np.linalg.norm(vectors[sample_idx], axis=1)
    bad = norms[np.abs(norms - 1.0) > _NORM_TOLERANCE]
    if bad.size:
        raise AssertionError(
            f"{video_id}: {bad.size} vector co norm ngoai "
            f"[1-{_NORM_TOLERANCE}, 1+{_NORM_TOLERANCE}]: {bad}"
        )


def _write_video_output(
    out_dir: Path, video_id: str, frame_records: list[tuple[int, Path]], vectors: np.ndarray
) -> None:
    vectors_dir = out_dir / "vectors"
    vectors_dir.mkdir(parents=True, exist_ok=True)

    jsonl_tmp = vectors_dir / f"{video_id}.jsonl.tmp"
    with jsonl_tmp.open("w", encoding="utf-8") as f:
        for frame_idx, _path in frame_records:
            keyframe_id = f"{video_id}_f{frame_idx}"
            record = {"keyframe_id": keyframe_id, "video_id": video_id, "frame_idx": frame_idx}
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    # Same np.save()-auto-suffix trap as the earlier scripts -- open the temp file as a file
    # object first so numpy never gets a chance to silently rewrite the ".tmp" name.
    npy_tmp = vectors_dir / f"{video_id}.npy.tmp"
    with npy_tmp.open("wb") as f:
        np.save(f, vectors)

    os.replace(npy_tmp, vectors_dir / f"{video_id}.npy")
    os.replace(jsonl_tmp, vectors_dir / f"{video_id}.jsonl")


def _cleanup_stale_tmp_files(vectors_dir: Path) -> None:
    if not vectors_dir.exists():
        return
    stale = sorted(vectors_dir.glob("*.tmp")) + sorted(vectors_dir.glob("*.tmp.npy"))
    for path in stale:
        # missing_ok=True: voi >1 shard, ca hai process goi ham nay gan nhu cung luc tren CUNG
        # vectors_dir khi resume -- shard kia co the da xoa file nay truoc, khong phai loi.
        path.unlink(missing_ok=True)
    if stale:
        print(f"Don {len(stale)} file tam con sot tu lan chay truoc: {[p.name for p in stale]}")


def main() -> None:
    args = _parse_args()
    if not (0 <= args.shard_id < args.num_shards):
        raise ValueError(f"--shard-id ({args.shard_id}) phai trong [0, --num-shards={args.num_shards})")
    shard_tag = f"[shard {args.shard_id}/{args.num_shards}]"

    # append, KHONG insert(0, ...): repo co package datasets/ trung ten thu vien HF
    # `datasets` -- insert o dau sys.path se khien `import datasets` bat nham package cua repo.
    sys.path.append(str(args.repo_dir))
    from app.encoders.pe_core import PeCoreEncoder  # noqa: E402 (needs sys.path append above)

    grouped = discover_keyframes(args.kf_root)
    video_ids = _select_video_ids(
        sorted(grouped), grouped, args.shard_id, args.num_shards, args.limit_videos, args.limit_keyframes
    )
    print(f"{shard_tag} nhan {len(video_ids)} video sau khi shard+limit.")

    # Shard nao cung goi ham nay tren CUNG mot vectors_dir -- an toan vi tap video_id giua cac
    # shard KHONG giao nhau ([shard_id::num_shards] tren mot list da khu trung), nen khong shard
    # nao xoa/dung tmp file cua shard khac.
    _cleanup_stale_tmp_files(args.out_dir / "vectors")

    encoder = PeCoreEncoder(
        hf_repo_id=args.hf_repo_id,
        device=args.device,
        batch_size=args.batch_size,
        model_id=args.model_id,
        preprocess_workers=args.preprocess_workers,
    )

    total_videos = len(video_ids)
    total_vectors_written = 0
    start_time = time.monotonic()

    for i, video_id in enumerate(video_ids, start=1):
        frame_records = grouped[video_id]
        if _is_video_done(args.out_dir, video_id, len(frame_records)):
            print(f"{shard_tag} [{i}/{total_videos}] {video_id}: da xong ({len(frame_records)} kf) -- bo qua (resume).")
            total_vectors_written += len(frame_records)
            continue

        video_start = time.monotonic()
        vectors = _embed_video(encoder, frame_records, args.chunk_size)
        if vectors.shape[0] != len(frame_records):
            raise AssertionError(
                f"{video_id}: so vector ({vectors.shape[0]}) != so keyframe ({len(frame_records)})"
            )
        _assert_normalized(vectors, video_id)
        _write_video_output(args.out_dir, video_id, frame_records, vectors)

        elapsed = time.monotonic() - video_start
        total_vectors_written += vectors.shape[0]
        print(f"{shard_tag} [{i}/{total_videos}] {video_id}: {vectors.shape[0]} kf trong {elapsed:.1f}s")

    total_elapsed = time.monotonic() - start_time
    rate = total_vectors_written / total_elapsed if total_elapsed > 0 else 0.0
    print(
        f"\n{shard_tag} Xong: {total_videos} video, {total_vectors_written} vector, "
        f"{total_elapsed:.1f}s ({total_elapsed / 60:.1f} phut), {rate:.1f} keyframe/s (rieng shard nay)."
    )


if __name__ == "__main__":
    main()


## 8. Smoke test ~1.000 keyframe

Chạy TRƯỚC khi treo job full 873 video — cho số đo throughput thật, thay vì ước lượng. Đây là
candidate nặng nhất trong plan; KHÔNG suy đoán từ số của SigLIP 2/BEiT-3.

**Chạy `NUM_SHARDS` process song song, mỗi process 1 GPU riêng** (`--device cuda:{shard_id}`)
— dùng `subprocess.Popen` (không phải `subprocess.run`) để không block nhau, rồi đợi cả hai
`.wait()` xong. `~1.000 keyframe` là **tổng** trên toàn bộ shard, chia đều qua
`--limit-keyframes` cho mỗi process (`1000 // NUM_SHARDS` mỗi bên) để throughput đo được là
throughput KẾT HỢP của cả 2 GPU, không phải của riêng một GPU.

In [ ]:
import subprocess
import time

_SMOKE_KEYFRAMES_TOTAL = 1000
_smoke_keyframes_per_shard = max(1, _SMOKE_KEYFRAMES_TOTAL // NUM_SHARDS)

smoke_start = time.monotonic()
smoke_procs = []
for _shard_id in range(NUM_SHARDS):
    smoke_procs.append(
        subprocess.Popen(
            [
                "python", "/kaggle/working/embed_pecore.py",
                "--kf-root", str(KF_ROOT),
                "--out-dir", str(OUT_DIR),
                "--repo-dir", str(REPO_DIR),
                "--hf-repo-id", HF_REPO_ID,
                "--model-id", MODEL_ID,
                "--chunk-size", str(CHUNK_SIZE),
                "--batch-size", str(BATCH_SIZE),
                "--preprocess-workers", str(PREPROCESS_WORKERS),
                "--device", f"cuda:{_shard_id}",
                "--num-shards", str(NUM_SHARDS),
                "--shard-id", str(_shard_id),
                "--limit-keyframes", str(_smoke_keyframes_per_shard),
            ],
            cwd=str(REPO_DIR),
        )
    )

_smoke_returncodes = [p.wait() for p in smoke_procs]
smoke_elapsed = time.monotonic() - smoke_start

_smoke_failed = [i for i, rc in enumerate(_smoke_returncodes) if rc != 0]
if _smoke_failed:
    raise RuntimeError(
        f"Shard(s) {_smoke_failed} loi trong smoke test (returncode != 0) -- xem output tren "
        "de biet shard nao va tai sao, dung lai truoc khi chay full."
    )

print(f"\nSmoke test ({NUM_SHARDS} shard song song): {smoke_elapsed:.1f}s")


In [ ]:
import numpy as np

smoke_npy_files = sorted(VECTORS_DIR.glob("*.npy"))
smoke_num_keyframes = int(sum(np.load(p).shape[0] for p in smoke_npy_files))
smoke_rate = smoke_num_keyframes / smoke_elapsed if smoke_elapsed > 0 else 0.0

first_video_id = smoke_npy_files[0].stem
first_vectors = np.load(smoke_npy_files[0])
norms = np.linalg.norm(first_vectors, axis=1)
print(f"{first_video_id}: shape={first_vectors.shape} norm min={norms.min():.5f} max={norms.max():.5f}")
assert abs(norms.min() - 1.0) < 1e-3 and abs(norms.max() - 1.0) < 1e-3, (
    "norm ngoai nguong -- dung lai, khong chay full 873 video truoc khi tim ra nguyen nhan"
)
assert first_vectors.shape[1] == EMBEDDING_DIM, (
    f"embedding_dim thuc te {first_vectors.shape[1]} != ky vong {EMBEDDING_DIM}"
)

print(
    f"\nSmoke test: {smoke_num_keyframes} keyframe trong {smoke_elapsed:.1f}s (wall-clock, "
    f"{NUM_SHARDS} GPU song song) -> {smoke_rate:.1f} keyframe/s KET HOP ca {NUM_SHARDS} GPU"
)
est_total_seconds = EXPECTED_NUM_KEYFRAMES / smoke_rate if smoke_rate > 0 else float("inf")
print(
    f"Uoc luong thoi gian full {EXPECTED_NUM_KEYFRAMES} keyframe (tuyen tinh theo mau smoke): "
    f"~{est_total_seconds / 3600:.1f} gio -- uoc luong tho, mat do kf lech nhieu giua cac "
    "batch, mau smoke co the khong dai dien cho toan corpus."
)
print(
    "\nKIEM TRA NGAN SACH: 30 gio GPU/tuan la CHUNG cho ca A1 (SigLIP 2) va A2 -- tru di gio "
    "A1 da dung truoc khi quyet dinh bam muc 9 (full run)."
)
print(
    "\nNeu GPU van ranh nhieu trong luc chay, thu tang PREPROCESS_WORKERS truoc, BATCH_SIZE "
    "sau -- xem comment o cell cau hinh."
)


## 9. Chạy full 873 video — cell nặng nhất

Chỉ bấm sau khi mục 6 (kiểm kê) và mục 8 (smoke test) đều xanh, bạn đã đọc số `keyframe/s` +
giờ ước tính, VÀ đã kiểm ngân sách 30 giờ GPU/tuần còn lại (trừ giờ A1 đã dùng). Nếu session
bị cắt giữa chừng, script tự resume ở lần chạy sau (bỏ qua video đã có đủ `.npy`/`.jsonl`).

**`NUM_SHARDS` process song song, mỗi process 1 GPU** — cùng cách launch với smoke test (mục
8), không giới hạn `--limit-keyframes`/`--limit-videos`. Giờ GPU tính vào ngân sách 30h/tuần
là **giờ mỗi GPU riêng** (2 GPU chạy 1 giờ = 2 giờ GPU-quota, không phải 1).

In [ ]:
full_run_start = time.monotonic()
full_procs = []
for _shard_id in range(NUM_SHARDS):
    full_procs.append(
        subprocess.Popen(
            [
                "python", "/kaggle/working/embed_pecore.py",
                "--kf-root", str(KF_ROOT),
                "--out-dir", str(OUT_DIR),
                "--repo-dir", str(REPO_DIR),
                "--hf-repo-id", HF_REPO_ID,
                "--model-id", MODEL_ID,
                "--chunk-size", str(CHUNK_SIZE),
                "--batch-size", str(BATCH_SIZE),
                "--preprocess-workers", str(PREPROCESS_WORKERS),
                "--device", f"cuda:{_shard_id}",
                "--num-shards", str(NUM_SHARDS),
                "--shard-id", str(_shard_id),
                "--limit-videos", "0",
                "--limit-keyframes", "0",
            ],
            cwd=str(REPO_DIR),
        )
    )

_full_returncodes = [p.wait() for p in full_procs]
full_run_elapsed = time.monotonic() - full_run_start

_full_failed = [i for i, rc in enumerate(_full_returncodes) if rc != 0]
if _full_failed:
    raise RuntimeError(
        f"Shard(s) {_full_failed} loi trong full run (returncode != 0) -- xem output tren de "
        "biet shard nao va tai sao. Video da xong (o CA HAI shard) van resume duoc binh thuong "
        "sau khi sua loi va chay lai cell nay."
    )

print(
    f"\nChay full ({NUM_SHARDS} GPU song song): {full_run_elapsed:.1f}s "
    f"({full_run_elapsed / 3600:.2f} gio wall-clock, ~{full_run_elapsed * NUM_SHARDS / 3600:.2f} "
    "gio GPU-quota)"
)


## 10. Kiểm tra output

Fail loud nếu còn rác `.tmp`/tên lạ, hoặc số hàng `.npy` lệch số dòng `.jsonl`, hoặc
`embedding_dim` không phải 1280.

In [ ]:
import re

_VIDEO_ID_RE = re.compile(r"^L\d+_V\d+$")

all_npy = sorted(VECTORS_DIR.glob("*.npy"))
npy_files = [p for p in all_npy if _VIDEO_ID_RE.match(p.stem)]
garbage = [p for p in all_npy if p not in npy_files]
if garbage:
    print(f"Rac con sot ({len(garbage)} file, ten khong khop video_id) -- nen xoa tay:")
    for p in garbage:
        print("  -", p.name)
    raise ValueError(
        f"{len(garbage)} file rac trong {VECTORS_DIR} -- xoa thu cong cac file tren roi chay "
        "lai cell nay (khong tu xoa o day de tranh xoa nham)."
    )

print(f"So video co output: {len(npy_files)} (ky vong {EXPECTED_NUM_VIDEOS})")

total_vectors = 0
mismatches = []
dim_mismatches = []
for npy_path in npy_files:
    vectors = np.load(npy_path)
    if vectors.shape[1] != EMBEDDING_DIM:
        dim_mismatches.append((npy_path.stem, vectors.shape[1]))
    jsonl_path = VECTORS_DIR / f"{npy_path.stem}.jsonl"
    with jsonl_path.open(encoding="utf-8") as f:
        num_lines = sum(1 for _ in f)
    if vectors.shape[0] != num_lines:
        mismatches.append((npy_path.stem, vectors.shape[0], num_lines))
    total_vectors += vectors.shape[0]

print(f"Tong vector: {total_vectors} (ky vong {EXPECTED_NUM_KEYFRAMES})")
if mismatches:
    raise AssertionError(f"{len(mismatches)} video lech so hang .npy / so dong .jsonl: {mismatches[:10]}")
if dim_mismatches:
    raise AssertionError(f"{len(dim_mismatches)} video co embedding_dim sai: {dim_mismatches[:10]}")
print(f"So hang .npy khop so dong .jsonl, embedding_dim == {EMBEDDING_DIM} cho toan bo video: OK")

!du -sh {OUT_DIR}


## 11. Image probe + text probe (manifest v2)

Image probe: chọn 16 `keyframe_id` cố định, trải đều theo chỉ số (không phải 16 cái đầu —
tránh dồn hết vào batch L21) để phủ nhiều batch, rồi trích thẳng vector từ `vectors/*.npy` đã
ghi (không encode lại). Text probe: encode lại 20 câu cố định, giống hệt hai notebook trước.

In [ ]:
import json

all_records = []  # (keyframe_id, video_id, frame_idx, row_index)
for jsonl_path in sorted(VECTORS_DIR.glob("*.jsonl")):
    video_id = jsonl_path.stem
    with jsonl_path.open(encoding="utf-8") as f:
        for row_index, line in enumerate(f):
            record = json.loads(line)
            all_records.append((record["keyframe_id"], video_id, record["frame_idx"], row_index))

if len(all_records) < IMAGE_PROBE_COUNT:
    raise ValueError(f"Chi co {len(all_records)} keyframe, khong du {IMAGE_PROBE_COUNT} cho image probe.")

# Trai deu theo chi so thay vi lay 16 cai dau (deu nam trong batch L21) -- de probe phu
# nhieu batch khac nhau.
step = len(all_records) // IMAGE_PROBE_COUNT
probe_records = [all_records[i * step] for i in range(IMAGE_PROBE_COUNT)]

probe_vectors = np.empty((IMAGE_PROBE_COUNT, EMBEDDING_DIM), dtype=np.float32)
probe_meta = []
for i, (keyframe_id, video_id, frame_idx, row_index) in enumerate(probe_records):
    vectors = np.load(VECTORS_DIR / f"{video_id}.npy")
    probe_vectors[i] = vectors[row_index]
    probe_meta.append({"keyframe_id": keyframe_id, "video_id": video_id, "frame_idx": frame_idx})

PROBE_DIR.mkdir(parents=True, exist_ok=True)
np.save(PROBE_DIR / "image_probe_vectors.npy", probe_vectors)
(PROBE_DIR / "image_probe.json").write_text(
    json.dumps(probe_meta, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Image probe: {IMAGE_PROBE_COUNT} keyframe, luu tai {PROBE_DIR}")
for m in probe_meta:
    print(" -", m["keyframe_id"])


In [ ]:
%%writefile /kaggle/working/probe_encode_pecore.py
"""Encode fixed probe strings with PeCoreEncoder -- parity-check artifact for local import.

Generated by notebooks/kaggle_embed_pecore.ipynb (%%writefile) -- not tracked in git.
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np


def _parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--repo-dir", type=Path, required=True)
    parser.add_argument("--hf-repo-id", required=True)
    parser.add_argument("--model-id", required=True)
    parser.add_argument("--probe-texts-path", type=Path, required=True)
    parser.add_argument("--out-npy-path", type=Path, required=True)
    parser.add_argument("--device", default="cuda")
    return parser.parse_args()


def main() -> None:
    args = _parse_args()
    sys.path.append(str(args.repo_dir))
    from app.encoders.pe_core import PeCoreEncoder  # noqa: E402

    probe_texts = json.loads(args.probe_texts_path.read_text(encoding="utf-8"))
    encoder = PeCoreEncoder(
        hf_repo_id=args.hf_repo_id,
        device=args.device,
        batch_size=len(probe_texts),
        model_id=args.model_id,
    )
    vectors = encoder.encode_text(probe_texts)
    np.save(args.out_npy_path, vectors.astype(np.float32))
    print(f"Da encode {len(probe_texts)} probe text -> {args.out_npy_path}, shape={vectors.shape}")


if __name__ == "__main__":
    main()


In [ ]:
PROBE_DIR.mkdir(parents=True, exist_ok=True)
(PROBE_DIR / "probe_texts.json").write_text(
    json.dumps(PROBE_TEXTS, ensure_ascii=False, indent=2), encoding="utf-8"
)

subprocess.run(
    [
        "python", "/kaggle/working/probe_encode_pecore.py",
        "--repo-dir", str(REPO_DIR),
        "--hf-repo-id", HF_REPO_ID,
        "--model-id", MODEL_ID,
        "--probe-texts-path", str(PROBE_DIR / "probe_texts.json"),
        "--out-npy-path", str(PROBE_DIR / "probe_text_vectors.npy"),
        "--device", "cuda",
    ],
    check=True, cwd=str(REPO_DIR),
)


## 12. Sinh `manifest.json` (v2)

Khác `siglip2`: không có `hf_revision` để pin (xem cảnh báo đầu notebook). Thay vào đó,
**resolve commit SHA hiện tại của "main" qua HF API ngay lúc chạy** và ghi vào manifest như
thông tin xuất xứ, kèm `"hf_revision_pinned": false` để không ai nhầm đây là một pin có hiệu
lực. Hash trực tiếp file `open_clip_model.safetensors` (file weight `.safetensors` thật sự tồn
tại trên repo, ưu tiên hơn `.bin`) qua `hf_hub_download` — cùng file, không đoán đường dẫn cache.

In [ ]:
import hashlib
import subprocess

from huggingface_hub import HfApi, hf_hub_download


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


git_sha = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=str(REPO_DIR), capture_output=True, text=True, check=True
).stdout.strip()

resolved_hf_revision = HfApi().model_info(HF_REPO_ID).sha
print(f"HF repo '{HF_REPO_ID}' 'main' resolved to commit: {resolved_hf_revision}")

weight_path = Path(
    hf_hub_download(
        repo_id=HF_REPO_ID, filename="open_clip_model.safetensors", revision=resolved_hf_revision
    )
)
weight_sha256 = _sha256(weight_path)

final_npy_files = sorted(VECTORS_DIR.glob("*.npy"))
final_num_videos = len(final_npy_files)
final_num_keyframes = int(sum(np.load(p).shape[0] for p in final_npy_files))

manifest = {
    "manifest_version": 2,
    "encoder_kind": "pe_core",
    "model_id": MODEL_ID,
    "embedding_dim": EMBEDDING_DIM,
    "hf_repo_id": HF_REPO_ID,
    "hf_revision": resolved_hf_revision,
    "hf_revision_pinned": False,  # xem canh bao dau notebook -- open_clip 3.3.0 khong ho tro
                                   # pin revision qua hf-hub loader, day chi la thong tin
                                   # xuat xu, khong phai mot pin co hieu luc chan.
    "weight_files_sha256": {"open_clip_model.safetensors": weight_sha256},
    "preprocess": (
        "image: open_clip preprocess transform doc tu open_clip_config.json preprocess_cfg "
        "(mean/std=0.5, bilinear, resize_mode=squash, 448x448) | "
        "text: open_clip tokenizer doc context_length=72 tu text_cfg, custom_text CLIP BPE "
        "(vocab_size=49408), English-only -- KHONG phai tokenizer da ngu nhu SigLIP 2"
    ),
    "library_versions": {
        "torch": torch.__version__,
        "open_clip": open_clip.__version__,
        "timm": timm.__version__,
    },
    "num_shards": NUM_SHARDS,  # thong tin xuat xu -- khong anh huong ket qua (video_id nao
                                # cung duoc dung dung 1 lan, bat ke chia may shard), chi giup
                                # truy vet throughput/gio GPU-quota da dung khi doc lai sau nay
    "num_videos": final_num_videos,
    "num_keyframes": final_num_keyframes,
    "git_sha": git_sha,
    "probe_text_count": len(PROBE_TEXTS),
    "image_probe": {
        "count": IMAGE_PROBE_COUNT,
        "keyframe_ids": [m["keyframe_id"] for m in probe_meta],
    },
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest, ensure_ascii=False, indent=2))

if final_num_videos != EXPECTED_NUM_VIDEOS or final_num_keyframes != EXPECTED_NUM_KEYFRAMES:
    print(
        "\nCANH BAO: num_videos/num_keyframes trong manifest KHONG khop ky vong "
        f"({EXPECTED_NUM_VIDEOS}/{EXPECTED_NUM_KEYFRAMES})."
    )


## 13. Mang dữ liệu về local + resume nếu session bị cắt

### Mang về local (khi mục 9 đã báo đủ 873/873 video)

1. Commit notebook version → tab **Output** → **Download All**
   (`embed_out_pecore/` — chỉ vector, không kèm ảnh).
2. Đặt vào `data/embed_out_pecore/` ở local.
3. Chờ `scripts/import_embed_out.py --encoder pe_core` + dispatch `"pe_core"` trong
   `app/encoders/factory.py` (plan §T1/§T5 — **CHƯA implement** tại thời điểm viết notebook
   này) — nó sẽ chạy cổng parity (text probe + image probe + hash weight) TRƯỚC khi ghi bất kỳ
   vector nào vào Milvus, và ghi vào collection `keyframes_pe_core_bigg14_448` — **không đụng**
   collection BEiT-3/SigLIP 2 hay MongoDB.
4. **Lưu ý cho ai viết cổng parity đó**: `manifest.json` của artifact này có
   `hf_revision_pinned: false` — không thể so sánh bằng-nhau-tuyệt-đối với một giá trị cấu hình
   local như `verify_hf_identity()` hiện làm cho SigLIP 2 (`pipelines/verify_embed_parity.py`),
   vì `configs/model/pe_core.yaml` không có trường `hf_revision` để so. Cổng an toàn thật sự
   cho candidate này là `weight_files_sha256` + `run_parity_probe()` (cosine ảnh + chữ), không
   phải so khớp revision.

### Nếu Kaggle cắt session giữa chừng, hoặc cần chạy tiếp phiên sau

1. Commit notebook version hiện tại (dù chạy dở) để `/kaggle/working/embed_out_pecore/` (phần
   đã xong) trở thành Output.
2. Mở session mới, chạy lại mục 1–7 (GPU/clone/dependency/config/tower-check/kiểm kê/sinh
   script) — các bước này idempotent.
3. **Add** chính Output của lần chạy trước làm Kaggle Dataset input mới, rồi copy nó đè lên
   `OUT_DIR` trước khi chạy lại mục 9:
   ```python
   import shutil
   PREVIOUS_SESSION_DATASET_DIR_NAME = ""  # ví dụ: "aic-embed-out-pecore-session1"
   if PREVIOUS_SESSION_DATASET_DIR_NAME:
       prev_dir = Path(f"/kaggle/input/{PREVIOUS_SESSION_DATASET_DIR_NAME}")
       shutil.copytree(prev_dir, OUT_DIR, dirs_exist_ok=True)
   ```
4. Chạy lại đúng cell ở mục 9 — video đã có đủ `.npy`/`.jsonl` khớp số dòng được bỏ qua tự
   động. Lặp lại tới khi mục 10 báo đủ 873/873.
5. Mục 11–12 (probe + manifest) chỉ cần chạy **một lần cuối cùng**, sau khi mục 10 đã xác nhận
   đủ 873/873 — không cần chạy lại mỗi phiên.
